# Свёрточный автокодировщик MNIST — версия 2.0

    **Цель:** Разобрать сжатие изображения и восстановление без классов на выходе.

    **Алгоритм:** Encoder использует Conv2D и pooling; decoder — Conv2DTranspose. Минимизируется MSE, не MAE.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('autoencoder-mnist-keras')


## Данные
В builtin-режиме загрузка MNIST разрешается только явным флагом.


In [ ]:
from ml_lab.runtime import tensorflow_setup, demo_images, load_mnist
tf = tensorflow_setup(cfg)
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
if cfg.source == 'demo':
    all_images, _ = demo_images(192, 28, 28, classes=4, seed=cfg.seed)
    x_train, x_val, x_test = all_images[:128], all_images[128:160], all_images[160:]
elif cfg.source == 'builtin':
    train, test = load_mnist(cfg, tf)
    train = train.astype('float32')[..., None] / 255
    test = test.astype('float32')[..., None] / 255
    if cfg.quick: train, test = train[:256], test[:64]
    boundary = int(len(train) * .8)
    x_train, x_val, x_test = train[:boundary], train[boundary:], test
else:
    raise ValueError('Для этого урока выберите demo или builtin')


## Encoder и decoder
Архитектура сохранена; обновлены optimizer и формат сохранения.


In [ ]:
def base_autoencoder(shape=(28, 28, 1)):
    inp = layers.Input(shape=shape)
    x = inp
    for filters in [32, 64]:
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
            x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    for filters in [64, 32]:
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        for _ in range(2):
            x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
            x = layers.BatchNormalization()(x)
    out = layers.Conv2D(shape[-1], 3, padding='same', activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.0001), loss='mean_squared_error')
    return model
model = base_autoencoder()
assert model.output_shape[1:] == (28, 28, 1)
history = model.fit(x_train, x_train, validation_data=(x_val, x_val), epochs=1 if cfg.quick else 100, batch_size=16 if cfg.quick else 256, verbose=0)
model.save_weights(cfg.output / 'autoencoder.weights.h5')
model.load_weights(cfg.output / 'autoencoder.weights.h5')


## Восстановление и MSE на test
Для метрики оставляем float [0, 1], не округляем в uint8.


In [ ]:
prediction = model.predict(x_test, verbose=0)
mse = np.mean((x_test - prediction) ** 2)
print('Test MSE:', float(mse))
plt.figure(figsize=(10, 4))
for i in range(5):
    for row, values in enumerate([x_test, prediction]):
        plt.subplot(2, 5, row * 5 + i + 1); plt.imshow(values[i, ..., 0], cmap='gray', vmin=0, vmax=1); plt.axis('off')
plt.show()


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Меняйте размер bottleneck.
2. Сравните MSE и MAE.
3. Добавьте шум только на вход и попробуйте denoising.
4. Не смешивайте нормированные float-данные и uint8 при оценке.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
